# Assignment 04 · Chương MLP đối kháng CNN và phân tích PCA không gian ẩn

**Sinh viên:** Nguyễn Duy Nghĩa · **Mã sinh viên:** B23DCCN600 · **Lớp:** D23CTPM01
**Học phần:** Phát triển các Hệ thống Thông minh · **Giảng viên:** PGS.TS Trần Đình Quế
**Học kỳ:** Học kỳ 1 năm học 2026 – 2027

---

## Mục tiêu của notebook

Toàn bộ các notebook trước trong Assignment 04 đều mặc nhiên chấp nhận rằng tích chập là lựa
chọn đúng cho dữ liệu ảnh. Notebook này đặt lại câu hỏi đó một cách sòng phẳng:

> **Tích chập có thực sự cần thiết hay không, hay một mạng dày đủ lớn cũng làm được điều tương tự?**

Báo cáo trả lời bằng hai phần độc lập nhưng bổ trợ cho nhau.

**Phần A. Thí nghiệm đối kháng có kiểm soát.** Chúng tôi huấn luyện một mạng truyền thẳng
(Multi-Layer Perceptron, viết tắt MLP) trên MNIST và CIFAR-10 dưới **đúng những điều kiện đã
dùng cho hai CNN anh em**: cùng cách chia tập, cùng hạt giống ngẫu nhiên 42, cùng hằng số chuẩn
hóa đọc trực tiếp từ tệp `*_preproc.json` của chúng, cùng hàm mất mát Cross-Entropy, cùng bộ tối
ưu Adam, cùng số epoch và cùng kích thước lô. Điểm mấu chốt về phương pháp luận: chúng tôi
**không huấn luyện lại CNN**, mà nạp đúng trọng số đã lưu của hai notebook `mnist` và `cifar10`.
Chỉ khi biến duy nhất thay đổi là kiến trúc thì chênh lệch quan sát được mới quy được cho kiến trúc.

**Phần B. Phân tích PCA không gian ẩn.** Chúng tôi nạp lại CNN PyTorch đã huấn luyện của mỗi
miền, gọi `extract_features(x)` trên toàn bộ 10.000 ảnh test để lấy véc-tơ ẩn 128 chiều ở tầng áp
chót, rồi chiếu xuống mặt phẳng hai chiều bằng PCA. Mục đích là quan sát xem mạng đã **tự tổ chức**
không gian biểu diễn như thế nào, và đặc biệt là trên CIFAR-10 có xuất hiện cấu trúc ngữ nghĩa cấp
cao nào không mặc dù hàm mất mát phạt mọi nhầm lẫn như nhau và không ai dạy mạng bất kỳ phân cấp nào.

Nguồn chân lý về tên tệp, tên khóa JSON và danh mục hình là `CONTRACT.md`, mục 4, 5 và 6.

## 1. Nhập thư viện và cố định hạt giống ngẫu nhiên

Tất cả nguồn ngẫu nhiên đều được cố định ở giá trị 42, đúng như mục 1 của hợp đồng tích hợp, để
lần chạy này tái lập được. Ngoài ra chúng tôi đặt `torch.set_num_threads(4)`: trên máy thực nghiệm,
số luồng mặc định làm mỗi epoch mất khoảng 40 giây trong khi 4 luồng chỉ mất khoảng 2,4 giây. Đây
là ghi chú kỹ thuật đã được đo trong phiên làm việc trước và được ghi lại trong `HANDOFF.md`.

In [1]:
import os, sys, json, time, math, importlib.util, inspect
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, silhouette_score
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Xem HANDOFF.md: 4 luong nhanh hon nhieu lan so voi mac dinh tren may nay.
torch.set_num_threads(4)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

FIG_DIR = '../reports/figures'
REP_DIR = '../reports'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

MNIST_DIR = '../../mnist'
CIFAR_DIR = '../../cifar10'

DEVICE = torch.device('cpu')

print('NumPy       :', np.__version__)
print('PyTorch     :', torch.__version__, '| CUDA kha dung:', torch.cuda.is_available())
print('Thiet bi    :', DEVICE, '| so luong CPU PyTorch =', torch.get_num_threads())
print('Hat giong   :', RANDOM_SEED)

NumPy       : 2.5.1
PyTorch     : 2.13.0+cu126 | CUDA kha dung: True
Thiet bi    : cpu | so luong CPU PyTorch = 4
Hat giong   : 42


## 2. Hai giả thuyết, phát biểu trước khi nhìn kết quả

Báo cáo phát biểu rõ hai giả thuyết ở đây, **trước** khi trình bày bất kỳ con số nào, để phần
kết quả phía sau là một phép kiểm chứng chứ không phải một lời kể lại đã biết trước đáp án.

**Giả thuyết H1 (MNIST): MLP sẽ bám khá sát CNN.**
Các chữ số trong MNIST đã được căn giữa sẵn theo trọng tâm khối mực, nằm trên nền đen đồng nhất,
và mọi ảnh đều có cùng một tỉ lệ chuẩn hóa. Nói cách khác, phương sai tịnh tiến, thứ mà tích chập
sinh ra để khai thác, gần như đã bị loại bỏ khỏi dữ liệu bởi chính quy trình tạo tập dữ liệu. Khi
mỗi chữ số gần như luôn xuất hiện ở cùng một vùng điểm ảnh, một tầng dày hoàn toàn có thể học
trực tiếp mối liên hệ giữa vị trí tuyệt đối và nhãn. Vì vậy chúng tôi dự đoán khoảng cách giữa
MLP và CNN trên MNIST là nhỏ, ở mức một vài phần trăm hoặc thấp hơn.

**Giả thuyết H2 (CIFAR-10): MLP sẽ tụt lại rất xa.**
Có hai lý do cộng hưởng. Thứ nhất, phép làm phẳng ảnh $32 \times 32 \times 3$ thành véc-tơ 3072
chiều phá vỡ quan hệ lân cận không gian: hai điểm ảnh kề nhau theo chiều dọc bị đẩy cách nhau 32
vị trí trong véc-tơ, và ba giá trị R, G, B của **cùng một điểm ảnh** bị tách thành ba ô cách nhau
1024 chỉ số nếu dữ liệu được xếp theo bố cục kênh trước. Với MLP, mọi hoán vị của 3072 chỉ số đầu
vào đều cho cùng một bài toán học, nên toàn bộ thông tin tô-pô của ảnh bị vứt bỏ ngay ở tầng đầu.
Thứ hai, khác với MNIST, đối tượng trong CIFAR-10 xuất hiện ở vị trí, tỉ lệ, tư thế và nền tùy ý,
nên phương sai tịnh tiến ở đây là rất lớn và chính là thứ mà chia sẻ trọng số của tích chập xử lý
hiệu quả. Chúng tôi dự đoán chênh lệch trên CIFAR-10 lớn hơn hẳn chênh lệch trên MNIST.

**Điểm kiểm soát về dung lượng mô hình.** Lập luận trên chỉ có giá trị nếu MLP không thua vì
thiếu tham số. Vì vậy notebook sẽ in tường minh số tham số của cả bốn mô hình và kiểm chứng bằng
số thật xem MLP có nhiều tham số hơn CNN hay không. Nếu MLP có nhiều tham số hơn mà vẫn thua, thì
nó thua vì **giả định kiến trúc** chứ không vì dung lượng.

## 3. Cơ sở toán học của phép so sánh

### 3.1 Đếm tham số: vì sao tích chập rẻ hơn rất nhiều

Một tầng dày nối $n_{\text{in}}$ nơ-ron vào $n_{\text{out}}$ nơ-ron có

$$P_{\text{dense}} = n_{\text{in}} \cdot n_{\text{out}} + n_{\text{out}}$$

tham số, tức là **tỉ lệ với kích thước ảnh đầu vào**. Trong khi đó một tầng tích chập hai chiều
biến $C_{\text{in}}$ kênh thành $C_{\text{out}}$ kênh với nhân $K \times K$ chỉ có

$$P_{\text{conv}} = C_{\text{in}} \cdot K^2 \cdot C_{\text{out}} + C_{\text{out}}$$

tham số, **hoàn toàn độc lập với chiều cao $H$ và chiều rộng $W$ của ảnh**. Bộ lọc được trượt và
dùng lại ở mọi vị trí, đó chính là cơ chế chia sẻ trọng số.

### 3.2 Bất biến tịnh tiến: giả định quy nạp mà MLP không có

Ký hiệu $T_t$ là phép tịnh tiến ảnh đi một khoảng $t$. Phép tích chập thỏa mãn tính tương biến

$$(T_t f) * w = T_t (f * w),$$

nghĩa là dịch chuyển đầu vào chỉ làm dịch chuyển bản đồ đặc trưng đầu ra chứ không thay đổi nội
dung của nó. Ghép thêm một tầng gộp cực đại, đặc trưng trở thành gần như bất biến với tịnh tiến
cục bộ. Một mẫu cạnh học được ở góc trên trái do đó dùng lại được ở góc dưới phải mà không tốn
thêm một tham số nào.

Với MLP, tình thế ngược lại. Gọi $\pi$ là một hoán vị bất kỳ của 3072 chỉ số đầu vào. Nếu ta hoán
vị đồng thời các cột của ma trận trọng số tầng đầu theo đúng $\pi$ thì hàm mà mạng biểu diễn
**không đổi**. Nói cách khác, MLP xem đầu vào như một túi các giá trị vô hướng không có tô-pô: nó
không hề biết rằng điểm ảnh $(i, j)$ và $(i, j+1)$ nằm cạnh nhau. Toàn bộ cấu trúc hai chiều mà
tích chập khai thác là thông tin mà phép làm phẳng đã xóa sạch.

Đây cũng chính là luận điểm đã được kiểm chứng định lượng ở chương dữ liệu bảng, nơi phép hoán vị
thứ tự 8 cột chỉ làm F1 đổi 0,68 phần trăm. Với dữ liệu bảng, việc không có tô-pô là **đúng bản
chất dữ liệu**. Với ảnh, đó là một tổn thất thông tin do kiến trúc tự gây ra.

### 3.3 Phân tích thành phần chính trên không gian ẩn

Gọi $z_i \in \mathbb{R}^{128}$ là véc-tơ ẩn mà `extract_features` trả về cho ảnh test thứ $i$, và
$\bar z$ là véc-tơ trung bình. Ma trận hiệp phương sai mẫu là

$$\Sigma = \frac{1}{N-1} \sum_{i=1}^{N} (z_i - \bar z)(z_i - \bar z)^{\top} \in \mathbb{R}^{128 \times 128}.$$

Khai triển trị riêng $\Sigma = U \Lambda U^{\top}$ với $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_{128}$.
Phép chiếu PCA hai chiều lấy hai véc-tơ riêng đầu tiên:

$$\tilde z_i = \big[\, u_1^{\top}(z_i - \bar z),\; u_2^{\top}(z_i - \bar z) \,\big] \in \mathbb{R}^2 .$$

Tỉ lệ phương sai được giải thích của trục thứ $k$ là $\lambda_k / \sum_{j=1}^{128} \lambda_j$.

**Cảnh báo diễn giải, cần đọc kỹ.** PCA là một phép chiếu **tuyến tính** và ở đây nó chỉ giữ lại
2 trong 128 hướng. Hệ quả có hai chiều và bất đối xứng:

1. Nếu hai nhóm **tách rời** trên hình chiếu thì chúng chắc chắn cũng tách rời trong không gian
   đầy đủ. Mức tách quan sát được là một **cận dưới** của mức tách thật.
2. Nếu hai nhóm **chồng lấn** trên hình chiếu thì điều đó **không chứng minh** chúng chồng lấn
   trong không gian 128 chiều. Rất có thể hướng phân tách nằm trong 126 hướng đã bị bỏ đi, hoặc
   là một mặt phân tách phi tuyến mà PCA về nguyên tắc không thể biểu diễn.

Vì vậy mọi kết luận rút ra từ hai hình PCA phía dưới đều được phát biểu theo hướng thận trọng, và
chúng tôi bổ sung một chỉ số định lượng thay vì chỉ tin vào mắt nhìn.

## 4. Hàm dùng chung cho cả hai miền

Định nghĩa MLP theo đúng mục 4 của hợp đồng tích hợp:
`Flatten → Dense(512) → Dense(256) → Dense(128) → Dense(10)` với `Dropout(0.2)` sau mỗi tầng ẩn.
Hàm huấn luyện dùng Adam, Cross-Entropy, in log từng epoch, và chọn trọng số tốt nhất theo
val loss. Tập test 10.000 ảnh không bao giờ tham gia vào việc chọn epoch, đúng như mục 3 của hợp đồng.

In [2]:
class MLP(nn.Module):
    """MLP doi khang theo hop dong: Flatten -> 512 -> 256 -> 128 -> 10, Dropout(0.2)."""

    def __init__(self, in_dim: int, n_classes: int = 10, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, 512), nn.ReLU(inplace=True), nn.Dropout(p_drop),
            nn.Linear(512, 256),    nn.ReLU(inplace=True), nn.Dropout(p_drop),
            nn.Linear(256, 128),    nn.ReLU(inplace=True), nn.Dropout(p_drop),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def count_params(model) -> int:
    """So tham so co the hoc duoc."""
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def evaluate(model, loader, criterion):
    """Tra ve (loss trung binh, accuracy, ma tran xac suat softmax)."""
    model.eval()
    tot_loss, correct, n, probs = 0.0, 0, 0, []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            tot_loss += criterion(out, yb).item() * len(yb)
            p = torch.softmax(out, dim=1)
            probs.append(p.numpy())
            correct += int((p.argmax(1) == yb).sum())
            n += len(yb)
    return tot_loss / n, correct / n, np.concatenate(probs, axis=0)


def train_mlp(model, train_ld, val_ld, epochs, lr=1e-3, tag=''):
    """Huan luyen MLP, in log tung epoch, giu lai trong so co val loss thap nhat."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_vl, best_state, best_epoch = float('inf'), None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        run_loss, correct, n = 0.0, 0, 0
        for xb, yb in train_ld:
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * len(yb)
            correct += int((out.argmax(1) == yb).sum())
            n += len(yb)
        tr_loss, tr_acc = run_loss / n, correct / n
        va_loss, va_acc, _ = evaluate(model, val_ld, criterion)
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(va_loss)
        hist['train_acc'].append(tr_acc);   hist['val_acc'].append(va_acc)
        flag = ''
        if va_loss < best_vl:
            best_vl, best_epoch = va_loss, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            flag = '  <- val loss tot nhat'
        print(f'[{tag}] epoch {ep:2d}/{epochs}  '
              f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  '
              f'val_loss={va_loss:.4f} val_acc={va_acc:.4f}{flag}')
    dt = time.time() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f'[{tag}] xong sau {dt:.1f}s, epoch tot nhat = {best_epoch}')
    return hist, best_epoch, dt


def load_preproc(path):
    """Doc hang so chuan hoa do notebook anh em ghi ra."""
    with open(path, 'r', encoding='utf-8') as f:
        p = json.load(f)
    mean = np.asarray(p['mean'], dtype=np.float32)
    std = np.asarray(p['std'], dtype=np.float32)
    scale = float(p.get('scale', 255.0))
    return p, mean, std, scale


def apply_preproc(x_u8, mean, std, scale):
    """x_u8 dang (N,H,W) hoac (N,H,W,C) -> tensor NCHW da chuan hoa."""
    x = x_u8.astype(np.float32) / scale
    if x.ndim == 3:
        x = x[..., None]                      # (N,H,W,1)
    if mean.size == 1:
        x = (x - float(mean)) / float(std)
    else:
        x = (x - mean.reshape(1, 1, 1, -1)) / std.reshape(1, 1, 1, -1)
    return np.ascontiguousarray(x.transpose(0, 3, 1, 2))   # NCHW


def load_cnn(def_path, weight_path):
    """Nap dinh nghia CNN cua notebook anh em roi do trong so da huan luyen vao.

    Tim lop nn.Module trong mo-dun mot cach dong de khong phu thuoc vao ten lop.
    """
    name = os.path.splitext(os.path.basename(def_path))[0]
    spec = importlib.util.spec_from_file_location(name, def_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    cands = [o for _, o in vars(mod).items()
             if inspect.isclass(o) and issubclass(o, nn.Module)
             and o is not nn.Module and o.__module__ == mod.__name__]
    if not cands:
        raise RuntimeError(f'Khong tim thay lop nn.Module nao trong {def_path}')
    cls = cands[0]
    try:
        model = cls()
    except TypeError:
        model = cls(n_classes=10)
    obj = torch.load(weight_path, map_location='cpu', weights_only=False)
    if isinstance(obj, nn.Module):
        model = obj
    else:
        sd = obj['state_dict'] if (isinstance(obj, dict) and 'state_dict' in obj) else obj
        model.load_state_dict(sd)
    model.eval()
    if not hasattr(model, 'extract_features'):
        raise RuntimeError(f'{cls.__name__} khong co phuong thuc extract_features')
    print(f'Da nap {cls.__name__} tu {weight_path}  ({count_params(model):,} tham so)')
    return model


@torch.no_grad()
def extract_all_features(model, X_nchw, batch=512):
    """Chay extract_features tren toan bo tap, tra ve mang (N, 128)."""
    model.eval()
    outs = []
    for i in range(0, len(X_nchw), batch):
        xb = torch.from_numpy(X_nchw[i:i + batch])
        outs.append(model.extract_features(xb).numpy())
    return np.concatenate(outs, axis=0)


print('Da dinh nghia xong cac ham dung chung.')

Da dinh nghia xong cac ham dung chung.


## 5. Phần A1: MLP trên MNIST

### 5.1 Nạp dữ liệu và tái lập đúng điều kiện của CNN anh em

Điều kiện kiểm soát được thực hiện cụ thể như sau. Chỉ số tách train và validation được sinh lại
bằng `train_test_split(test_size=0.2, stratify=y, random_state=42)`, đúng công thức mà notebook
`mnist` đã dùng, nên hai bên nhận **cùng một tập ảnh** cho mỗi nhánh. Hằng số `mean` và `std`
không được tính lại theo ý riêng mà **đọc thẳng từ `mnist/models/mnist_preproc.json`**. Để chắc
chắn hai bên thực sự trùng nhau, notebook tự tính lại hằng số từ nhánh train của mình rồi so sánh
với giá trị trong tệp và in ra sai lệch tuyệt đối.

In [3]:
d = np.load(f'{MNIST_DIR}/data/mnist.npz')
x_tr_raw_m, y_tr_raw_m = d['x_train'], d['y_train'].astype(np.int64)
x_te_raw_m, y_te_raw_m = d['x_test'],  d['y_test'].astype(np.int64)

print('MNIST tho     :', x_tr_raw_m.shape, x_te_raw_m.shape, '| dtype', x_tr_raw_m.dtype)
print('So lop        :', len(np.unique(y_tr_raw_m)))
print('Phan phoi test:', np.bincount(y_te_raw_m, minlength=10).tolist())

idx_all = np.arange(len(x_tr_raw_m))
idx_tr_m, idx_va_m = train_test_split(idx_all, test_size=0.2,
                                      stratify=y_tr_raw_m, random_state=RANDOM_SEED)

pp_m, MEAN_M, STD_M, SCALE_M = load_preproc(f'{MNIST_DIR}/models/mnist_preproc.json')
print('\nHang so chuan hoa doc tu mnist_preproc.json:')
print('  mean =', MEAN_M.tolist(), ' std =', STD_M.tolist(), ' scale =', SCALE_M)

# Kiem chung cheo: tinh lai tu nhanh train cua chinh notebook nay.
_chk = x_tr_raw_m[idx_tr_m].astype(np.float32) / SCALE_M
print(f'  tinh lai tai cho: mean = {_chk.mean():.10f}  std = {_chk.std():.10f}')
print(f'  sai lech tuyet doi: mean {abs(float(_chk.mean()) - float(MEAN_M)):.3e}, '
      f'std {abs(float(_chk.std()) - float(STD_M)):.3e}')
print(f'  so anh train {len(idx_tr_m)} (tep ghi {pp_m.get("n_train")}), '
      f'val {len(idx_va_m)} (tep ghi {pp_m.get("n_val")})')
del _chk

Xtr_m = apply_preproc(x_tr_raw_m[idx_tr_m], MEAN_M, STD_M, SCALE_M); ytr_m = y_tr_raw_m[idx_tr_m]
Xva_m = apply_preproc(x_tr_raw_m[idx_va_m], MEAN_M, STD_M, SCALE_M); yva_m = y_tr_raw_m[idx_va_m]
Xte_m = apply_preproc(x_te_raw_m,           MEAN_M, STD_M, SCALE_M); yte_m = y_te_raw_m

print('\nSau chuan hoa:', Xtr_m.shape, Xva_m.shape, Xte_m.shape)
print(f'Khoang gia tri train: [{Xtr_m.min():.4f}, {Xtr_m.max():.4f}]')

MNIST tho     : (60000, 28, 28) (10000, 28, 28) | dtype uint8
So lop        : 10
Phan phoi test: [980, 1135, 1032, 1010, 982, 892, 958, 1028, 974, 1009]



Hang so chuan hoa doc tu mnist_preproc.json:
  mean = 0.1307886838912964  std = 0.3082403540611267  scale = 255.0
  tinh lai tai cho: mean = 0.1307886839  std = 0.3082403541


  sai lech tuyet doi: mean 0.000e+00, std 0.000e+00
  so anh train 48000 (tep ghi 48000), val 12000 (tep ghi 12000)



Sau chuan hoa: (48000, 1, 28, 28) (12000, 1, 28, 28) (10000, 1, 28, 28)
Khoang gia tri train: [-0.4243, 2.8199]


In [4]:
# Doc cau hinh huan luyen cua CNN anh em de dung lai y het.
with open(f'{MNIST_DIR}/reports/metrics_mnist.json', 'r', encoding='utf-8') as f:
    met_mnist = json.load(f)

cnn_m = met_mnist['models']['pytorch']
EPOCHS_M = int(cnn_m['epochs'])
BATCH_M  = 128          # BATCH_FW trong mnist/notebooks/02_mnist_frameworks_comparison.ipynb

print('CNN PyTorch cua mien mnist (doc tu metrics_mnist.json, KHONG huan luyen lai):')
print(f'  accuracy = {cnn_m["accuracy"]:.4f}   loss = {cnn_m["loss"]:.6f}   '
      f'tham so = {cnn_m["params"]:,}   epochs = {cnn_m["epochs"]}')
print(f'\nMLP se dung dung {EPOCHS_M} epoch, batch {BATCH_M}, Adam lr=1e-3, CrossEntropyLoss.')

CNN PyTorch cua mien mnist (doc tu metrics_mnist.json, KHONG huan luyen lai):
  accuracy = 0.9906   loss = 0.027565   tham so = 421,738   epochs = 10

MLP se dung dung 10 epoch, batch 128, Adam lr=1e-3, CrossEntropyLoss.


In [5]:
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

tr_ds = TensorDataset(torch.from_numpy(Xtr_m), torch.from_numpy(ytr_m))
va_ds = TensorDataset(torch.from_numpy(Xva_m), torch.from_numpy(yva_m))
te_ds = TensorDataset(torch.from_numpy(Xte_m), torch.from_numpy(yte_m))

g = torch.Generator(); g.manual_seed(RANDOM_SEED)
tr_ld_m = DataLoader(tr_ds, batch_size=BATCH_M, shuffle=True, generator=g)
va_ld_m = DataLoader(va_ds, batch_size=512, shuffle=False)
te_ld_m = DataLoader(te_ds, batch_size=512, shuffle=False)

IN_DIM_M = int(np.prod(Xtr_m.shape[1:]))
mlp_m = MLP(IN_DIM_M).to(DEVICE)
PARAMS_MLP_M = count_params(mlp_m)

print(f'MLP MNIST: dau vao {Xtr_m.shape[1:]} -> lam phang {IN_DIM_M} chieu')
print(f'  tham so MLP = {PARAMS_MLP_M:,}')
print(f'  tham so CNN = {cnn_m["params"]:,}')
print(f'  ti le MLP/CNN = {PARAMS_MLP_M / cnn_m["params"]:.2f}x')
print(mlp_m)

MLP MNIST: dau vao (1, 28, 28) -> lam phang 784 chieu
  tham so MLP = 567,434
  tham so CNN = 421,738
  ti le MLP/CNN = 1.35x
MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU(inplace=True)
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=256, out_features=128, bias=True)
    (8): ReLU(inplace=True)
    (9): Dropout(p=0.2, inplace=False)
    (10): Linear(in_features=128, out_features=10, bias=True)
  )
)


### 5.2 Huấn luyện MLP trên MNIST

In [6]:
hist_m, best_ep_m, time_m = train_mlp(mlp_m, tr_ld_m, va_ld_m,
                                      epochs=EPOCHS_M, lr=1e-3, tag='MNIST-MLP')

[MNIST-MLP] epoch  1/10  train_loss=0.3354 train_acc=0.8967  val_loss=0.1371 val_acc=0.9574  <- val loss tot nhat


[MNIST-MLP] epoch  2/10  train_loss=0.1356 train_acc=0.9594  val_loss=0.0955 val_acc=0.9714  <- val loss tot nhat


[MNIST-MLP] epoch  3/10  train_loss=0.1019 train_acc=0.9690  val_loss=0.0976 val_acc=0.9724


[MNIST-MLP] epoch  4/10  train_loss=0.0823 train_acc=0.9753  val_loss=0.0903 val_acc=0.9743  <- val loss tot nhat


[MNIST-MLP] epoch  5/10  train_loss=0.0677 train_acc=0.9794  val_loss=0.0824 val_acc=0.9762  <- val loss tot nhat


[MNIST-MLP] epoch  6/10  train_loss=0.0584 train_acc=0.9820  val_loss=0.0847 val_acc=0.9776


[MNIST-MLP] epoch  7/10  train_loss=0.0548 train_acc=0.9826  val_loss=0.0887 val_acc=0.9753


[MNIST-MLP] epoch  8/10  train_loss=0.0477 train_acc=0.9849  val_loss=0.0845 val_acc=0.9790


[MNIST-MLP] epoch  9/10  train_loss=0.0421 train_acc=0.9863  val_loss=0.0864 val_acc=0.9800


[MNIST-MLP] epoch 10/10  train_loss=0.0413 train_acc=0.9867  val_loss=0.0913 val_acc=0.9770
[MNIST-MLP] xong sau 18.5s, epoch tot nhat = 5


In [7]:
crit = nn.CrossEntropyLoss()
loss_mlp_m, acc_mlp_m, probs_mlp_m = evaluate(mlp_m, te_ld_m, crit)
pred_mlp_m = probs_mlp_m.argmax(1)
cm_mlp_m = confusion_matrix(yte_m, pred_mlp_m, labels=list(range(10)))

gap_m = cnn_m['accuracy'] - acc_mlp_m

print('=== MNIST, tap test 10.000 anh ===')
print(f'MLP : loss = {loss_mlp_m:.6f}   accuracy = {acc_mlp_m:.4f}   tham so = {PARAMS_MLP_M:,}')
print(f'CNN : loss = {cnn_m["loss"]:.6f}   accuracy = {cnn_m["accuracy"]:.4f}   '
      f'tham so = {cnn_m["params"]:,}')
print(f'Chenh lech accuracy (CNN - MLP) = {gap_m:.4f}  tuc {gap_m * 100:.2f} diem phan tram')

=== MNIST, tap test 10.000 anh ===
MLP : loss = 0.077014   accuracy = 0.9763   tham so = 567,434
CNN : loss = 0.027565   accuracy = 0.9906   tham so = 421,738
Chenh lech accuracy (CNN - MLP) = 0.0143  tuc 1.43 diem phan tram


In [8]:
_more = PARAMS_MLP_M > cnn_m['params']
display(Markdown(f"""
**Diễn giải kết quả MNIST.**
MLP đạt accuracy {acc_mlp_m*100:.2f} phần trăm trên 10.000 ảnh test, còn CNN PyTorch đã huấn luyện
sẵn của miền `mnist` đạt {cnn_m['accuracy']*100:.2f} phần trăm. Chênh lệch là
**{gap_m*100:.2f} điểm phần trăm**, tương ứng với {abs(gap_m)*10000:.0f} ảnh trong tổng số 10.000 ảnh test.

Về dung lượng, MLP có {PARAMS_MLP_M:,} tham số so với {cnn_m['params']:,} tham số của CNN, tức là
{'nhiều hơn' if _more else 'ít hơn'} khoảng {PARAMS_MLP_M / cnn_m['params']:.2f} lần.
{'Như vậy MLP không hề bị bỏ đói tham số.' if _more else 'Ở đây MLP có ít tham số hơn CNN nên phần chênh lệch chưa quy hoàn toàn cho kiến trúc được.'}

Con số này ủng hộ giả thuyết H1. MNIST là tập dữ liệu đã được căn giữa và chuẩn hóa tỉ lệ từ
trước, nên phần phương sai tịnh tiến mà tích chập sinh ra để xử lý gần như không còn. Một mạng
dày học trực tiếp ánh xạ từ vị trí điểm ảnh tuyệt đối sang nhãn vẫn đủ tốt, và khoảng cách còn
lại {gap_m*100:.2f} điểm phần trăm là phần lợi thế nhỏ mà tích chập thu được từ những biến thiên
nét chữ còn sót lại.
"""))


**Diễn giải kết quả MNIST.**
MLP đạt accuracy 97.63 phần trăm trên 10.000 ảnh test, còn CNN PyTorch đã huấn luyện
sẵn của miền `mnist` đạt 99.06 phần trăm. Chênh lệch là
**1.43 điểm phần trăm**, tương ứng với 143 ảnh trong tổng số 10.000 ảnh test.

Về dung lượng, MLP có 567,434 tham số so với 421,738 tham số của CNN, tức là
nhiều hơn khoảng 1.35 lần.
Như vậy MLP không hề bị bỏ đói tham số.

Con số này ủng hộ giả thuyết H1. MNIST là tập dữ liệu đã được căn giữa và chuẩn hóa tỉ lệ từ
trước, nên phần phương sai tịnh tiến mà tích chập sinh ra để xử lý gần như không còn. Một mạng
dày học trực tiếp ánh xạ từ vị trí điểm ảnh tuyệt đối sang nhãn vẫn đủ tốt, và khoảng cách còn
lại 1.43 điểm phần trăm là phần lợi thế nhỏ mà tích chập thu được từ những biến thiên
nét chữ còn sót lại.


## 6. Phần A2: MLP trên CIFAR-10

### 6.1 Chờ miền `cifar10`, và chặn theo chất lượng chứ không chỉ theo sự tồn tại của tệp

Miền `cifar10` được dựng song song bởi một tiến trình khác. Notebook này **không tải lại dữ liệu
và không huấn luyện lại CNN**, vì việc dùng đúng mô hình đã huấn luyện của miền kia chính là điều
làm cho phép so sánh có tính kiểm soát.

Ở đây có một cái bẫy về phương pháp đáng ghi lại, vì lần chạy đầu tiên của notebook này đã sập
đúng vào nó. Quy trình bên miền `cifar10` dựng đường ống ở quy mô rất nhỏ trước để kiểm thử
(2000 ảnh train, 2 epoch), ghi ra đủ cả năm tệp, rồi mới huấn luyện thật và ghi đè lên. Một phép
chờ chỉ kiểm tra **sự tồn tại của tệp** sẽ mở cổng ngay tại bản chạy thử đó. Hậu quả không hề nhẹ:
checkpoint chạy thử chỉ đạt 26,82 phần trăm, tức là gần mức đoán mò 10 phần trăm, nên không gian
ẩn 128 chiều về cơ bản là nhiễu, chỉ số tách nhóm phương tiện và động vật trở thành vô nghĩa, và
tệ nhất là MLP lại **thắng** CNN 19,52 điểm, cho ra một kết luận ngược hoàn toàn với thực tế.

Bài học tổng quát: **một điều kiện chờ phải kiểm tra thứ ta thực sự cần, chứ không phải thứ dễ
kiểm tra nhất.** Tệp tồn tại là điều kiện cần nhưng không đủ. Vì vậy cổng chặn dưới đây kiểm tra
đồng thời ba dấu hiệu của một lần chạy thật, bất kỳ dấu hiệu nào không đạt cũng giữ notebook lại:

1. `models.pytorch.accuracy` vượt ngưỡng 0,60, trong khi bản chạy thử chỉ đạt 0,2682;
2. `dataset.n_train` đạt ít nhất 30.000, trong khi bản chạy thử chỉ dùng 2.000;
3. `models.pytorch.epochs` đạt ít nhất 5, trong khi bản chạy thử chỉ chạy 2 epoch.

Ngoài ra kích thước cả năm tệp phải ổn định qua một chu kỳ thăm dò, để không đọc phải tệp đang
được ghi dở. Mọi số liệu của CNN dùng trong chương này đều được đọc **sau khi** cổng mở, không có
giá trị nào được nhớ lại từ trước đó.

In [9]:
NEED_CIFAR = [
    f'{CIFAR_DIR}/data/cifar10.npz',
    f'{CIFAR_DIR}/models/cifar10_cnn_pytorch.pt',
    f'{CIFAR_DIR}/models/cifar10_cnn_def.py',
    f'{CIFAR_DIR}/models/cifar10_preproc.json',
    f'{CIFAR_DIR}/reports/metrics_cifar10.json',
]

MIN_CNN_ACC = 0.60     # ban chay thu 2 epoch chi dat 0.2682, chay that phai vuot xa muc nay
MIN_N_TRAIN = 30000    # nhanh train day du la 40000; ban chay thu chi dung 2000
MIN_EPOCHS  = 5        # ban chay thu chi chay 2 epoch
MAX_WAIT_S  = 5400     # tran cho toi da 90 phut
POLL_S      = 60

METRICS_CIFAR = f'{CIFAR_DIR}/reports/metrics_cifar10.json'


def cifar_gate():
    """Tra ve (san_sang, ly_do). Chan theo CHAT LUONG chu khong chi theo su ton tai cua tep."""
    missing = [p for p in NEED_CIFAR if not os.path.exists(p)]
    if missing:
        return False, 'thieu ' + ', '.join(os.path.basename(p) for p in missing)
    try:
        with open(METRICS_CIFAR, 'r', encoding='utf-8') as f:
            m = json.load(f)
        pt = m['models']['pytorch']
        acc, ep = float(pt['accuracy']), int(pt['epochs'])
        ntr = int(m['dataset']['n_train'])
    except Exception as exc:
        return False, f'chua doc duoc metrics ({type(exc).__name__})'
    if acc <= MIN_CNN_ACC:
        return False, f'accuracy CNN = {acc:.4f} <= nguong {MIN_CNN_ACC} (con la ban chay thu)'
    if ntr < MIN_N_TRAIN:
        return False, f'n_train = {ntr:,} < {MIN_N_TRAIN:,} (con la ban chay thu)'
    if ep < MIN_EPOCHS:
        return False, f'epochs = {ep} < {MIN_EPOCHS} (con la ban chay thu)'
    return True, f'accuracy = {acc:.4f}, n_train = {ntr:,}, epochs = {ep}'


def _sig(paths):
    return tuple(os.path.getsize(p) if os.path.exists(p) else -1 for p in paths)

t_start = time.time()
prev = None
while True:
    ok, why = cifar_gate()
    if ok:
        cur = _sig(NEED_CIFAR)
        if cur == prev:                      # kich thuoc on dinh qua mot chu ky
            print(f'\nCONG MO sau {time.time() - t_start:.0f}s cho. Lan chay that: {why}')
            break
        prev = cur
        print(f'[{time.time() - t_start:6.0f}s] dat chat luong ({why}), '
              f'cho them {POLL_S}s de xac nhan kich thuoc tep on dinh...')
    else:
        prev = None
        print(f'[{time.time() - t_start:6.0f}s] chua qua cong: {why}')
    if time.time() - t_start > MAX_WAIT_S:
        raise TimeoutError(f'Qua {MAX_WAIT_S}s cho ban chay that cua mien cifar10. Ly do cuoi: {why}')
    time.sleep(POLL_S)

for p in NEED_CIFAR:
    print(f'  {os.path.basename(p):28s} {os.path.getsize(p):>12,} byte')

[     0s] dat chat luong (accuracy = 0.7758, n_train = 40,000, epochs = 25), cho them 60s de xac nhan kich thuoc tep on dinh...



CONG MO sau 60s cho. Lan chay that: accuracy = 0.7758, n_train = 40,000, epochs = 25
  cifar10.npz                   169,596,192 byte
  cifar10_cnn_pytorch.pt            764,657 byte
  cifar10_cnn_def.py                  3,320 byte
  cifar10_preproc.json                1,299 byte
  metrics_cifar10.json               31,387 byte


### 6.2 Nạp CIFAR-10 và tái lập đúng điều kiện tiền xử lý

Quy trình giống hệt phần MNIST: sinh lại chỉ số tách bằng cùng công thức, và đọc hằng số chuẩn
hóa trực tiếp từ `cifar10/models/cifar10_preproc.json` thay vì tự tính. Điểm khác duy nhất là
CIFAR-10 có ba kênh màu, nên hằng số chuẩn hóa có thể là một số vô hướng hoặc một véc-tơ ba
thành phần; hàm `apply_preproc` xử lý được cả hai trường hợp.

In [10]:
dc = np.load(f'{CIFAR_DIR}/data/cifar10.npz')
x_tr_raw_c = dc['x_train']
y_tr_raw_c = dc['y_train'].astype(np.int64).ravel()
x_te_raw_c = dc['x_test']
y_te_raw_c = dc['y_test'].astype(np.int64).ravel()

print('CIFAR-10 tho  :', x_tr_raw_c.shape, x_te_raw_c.shape, '| dtype', x_tr_raw_c.dtype)
print('Phan phoi test:', np.bincount(y_te_raw_c, minlength=10).tolist())

idx_all_c = np.arange(len(x_tr_raw_c))
idx_tr_c, idx_va_c = train_test_split(idx_all_c, test_size=0.2,
                                      stratify=y_tr_raw_c, random_state=RANDOM_SEED)

pp_c, MEAN_C, STD_C, SCALE_C = load_preproc(f'{CIFAR_DIR}/models/cifar10_preproc.json')
print('\nHang so chuan hoa doc tu cifar10_preproc.json:')
print('  mean =', np.asarray(MEAN_C).ravel().tolist())
print('  std  =', np.asarray(STD_C).ravel().tolist())
print('  scale =', SCALE_C)

_chk = x_tr_raw_c[idx_tr_c].astype(np.float32) / SCALE_C
print(f'  tinh lai tai cho (vo huong): mean = {_chk.mean():.10f}  std = {_chk.std():.10f}')
print(f'  tinh lai tai cho (theo kenh): mean = {_chk.mean(axis=(0,1,2)).tolist()}')
print(f'                                 std  = {_chk.std(axis=(0,1,2)).tolist()}')
del _chk

Xtr_c = apply_preproc(x_tr_raw_c[idx_tr_c], MEAN_C, STD_C, SCALE_C); ytr_c = y_tr_raw_c[idx_tr_c]
Xva_c = apply_preproc(x_tr_raw_c[idx_va_c], MEAN_C, STD_C, SCALE_C); yva_c = y_tr_raw_c[idx_va_c]
Xte_c = apply_preproc(x_te_raw_c,           MEAN_C, STD_C, SCALE_C); yte_c = y_te_raw_c

print('\nSau chuan hoa:', Xtr_c.shape, Xva_c.shape, Xte_c.shape)
print(f'Khoang gia tri train: [{Xtr_c.min():.4f}, {Xtr_c.max():.4f}]')
assert abs(float(Xtr_c.min())) < 20 and abs(float(Xtr_c.max())) < 20, \
    'Khoang gia tri sau chuan hoa bat thuong, kiem tra lai cifar10_preproc.json'

CIFAR_CLASSES = ['may bay', 'o to', 'chim', 'meo', 'huou',
                 'cho', 'ech', 'ngua', 'tau thuy', 'xe tai']
VEHICLE_IDX = [0, 1, 8, 9]              # airplane, automobile, ship, truck
ANIMAL_IDX  = [2, 3, 4, 5, 6, 7]        # bird, cat, deer, dog, frog, horse

CIFAR-10 tho  : (50000, 32, 32, 3) (10000, 32, 32, 3) | dtype uint8
Phan phoi test: [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]

Hang so chuan hoa doc tu cifar10_preproc.json:
  mean = [0.4910935163497925, 0.48214712738990784, 0.4465753734111786]
  std  = [0.24700766801834106, 0.24354353547096252, 0.26164260506629944]
  scale = 255.0


  tinh lai tai cho (vo huong): mean = 0.4732720256  std = 0.2515897453


  tinh lai tai cho (theo kenh): mean = [0.40959998965263367, 0.40959998965263367, 0.40959998965263367]


                                 std  = [0.24523046612739563, 0.24006126821041107, 0.24823197722434998]



Sau chuan hoa: (40000, 3, 32, 32) (10000, 3, 32, 32) (10000, 3, 32, 32)
Khoang gia tri train: [-1.9882, 2.1263]


In [11]:
with open(f'{CIFAR_DIR}/reports/metrics_cifar10.json', 'r', encoding='utf-8') as f:
    met_cifar = json.load(f)

cnn_c = met_cifar['models']['pytorch']
EPOCHS_C = int(cnn_c['epochs'])
BATCH_C  = 128

print('CNN PyTorch cua mien cifar10 (doc tu metrics_cifar10.json, KHONG huan luyen lai):')
print(f'  accuracy = {cnn_c["accuracy"]:.4f}   loss = {cnn_c["loss"]:.6f}   '
      f'tham so = {cnn_c["params"]:,}   epochs = {cnn_c["epochs"]}')
print(f'\nMLP se dung dung {EPOCHS_C} epoch, batch {BATCH_C}, Adam lr=1e-3, CrossEntropyLoss.')

CNN PyTorch cua mien cifar10 (doc tu metrics_cifar10.json, KHONG huan luyen lai):
  accuracy = 0.7758   loss = 0.652900   tham so = 188,970   epochs = 25

MLP se dung dung 25 epoch, batch 128, Adam lr=1e-3, CrossEntropyLoss.


In [12]:
torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

tr_ds_c = TensorDataset(torch.from_numpy(Xtr_c), torch.from_numpy(ytr_c))
va_ds_c = TensorDataset(torch.from_numpy(Xva_c), torch.from_numpy(yva_c))
te_ds_c = TensorDataset(torch.from_numpy(Xte_c), torch.from_numpy(yte_c))

g2 = torch.Generator(); g2.manual_seed(RANDOM_SEED)
tr_ld_c = DataLoader(tr_ds_c, batch_size=BATCH_C, shuffle=True, generator=g2)
va_ld_c = DataLoader(va_ds_c, batch_size=512, shuffle=False)
te_ld_c = DataLoader(te_ds_c, batch_size=512, shuffle=False)

IN_DIM_C = int(np.prod(Xtr_c.shape[1:]))
mlp_c = MLP(IN_DIM_C).to(DEVICE)
PARAMS_MLP_C = count_params(mlp_c)

print(f'MLP CIFAR-10: dau vao {Xtr_c.shape[1:]} -> lam phang {IN_DIM_C} chieu')
print(f'  tham so MLP = {PARAMS_MLP_C:,}')
print(f'  tham so CNN = {cnn_c["params"]:,}')
print(f'  ti le MLP/CNN = {PARAMS_MLP_C / cnn_c["params"]:.2f}x')

MLP CIFAR-10: dau vao (3, 32, 32) -> lam phang 3072 chieu
  tham so MLP = 1,738,890
  tham so CNN = 188,970
  ti le MLP/CNN = 9.20x


### 6.3 Huấn luyện MLP trên CIFAR-10

In [13]:
hist_c, best_ep_c, time_c = train_mlp(mlp_c, tr_ld_c, va_ld_c,
                                      epochs=EPOCHS_C, lr=1e-3, tag='CIFAR-MLP')

[CIFAR-MLP] epoch  1/25  train_loss=1.7863 train_acc=0.3600  val_loss=1.6333 val_acc=0.4207  <- val loss tot nhat


[CIFAR-MLP] epoch  2/25  train_loss=1.6038 train_acc=0.4316  val_loss=1.5394 val_acc=0.4560  <- val loss tot nhat


[CIFAR-MLP] epoch  3/25  train_loss=1.5224 train_acc=0.4624  val_loss=1.4904 val_acc=0.4735  <- val loss tot nhat


[CIFAR-MLP] epoch  4/25  train_loss=1.4612 train_acc=0.4846  val_loss=1.4733 val_acc=0.4839  <- val loss tot nhat


[CIFAR-MLP] epoch  5/25  train_loss=1.4085 train_acc=0.5012  val_loss=1.4344 val_acc=0.4967  <- val loss tot nhat


...

[CIFAR-MLP] epoch 21/25  train_loss=0.9729 train_acc=0.6523  val_loss=1.4122 val_acc=0.5414


[CIFAR-MLP] epoch 22/25  train_loss=0.9518 train_acc=0.6607  val_loss=1.4201 val_acc=0.5442


[CIFAR-MLP] epoch 23/25  train_loss=0.9323 train_acc=0.6675  val_loss=1.4242 val_acc=0.5364


[CIFAR-MLP] epoch 24/25  train_loss=0.9244 train_acc=0.6694  val_loss=1.4066 val_acc=0.5430


[CIFAR-MLP] epoch 25/25  train_loss=0.9142 train_acc=0.6732  val_loss=1.4075 val_acc=0.5438
[CIFAR-MLP] xong sau 71.3s, epoch tot nhat = 15


In [14]:
loss_mlp_c, acc_mlp_c, probs_mlp_c = evaluate(mlp_c, te_ld_c, crit)
pred_mlp_c = probs_mlp_c.argmax(1)
cm_mlp_c = confusion_matrix(yte_c, pred_mlp_c, labels=list(range(10)))

gap_c = cnn_c['accuracy'] - acc_mlp_c

print('=== CIFAR-10, tap test 10.000 anh ===')
print(f'MLP : loss = {loss_mlp_c:.6f}   accuracy = {acc_mlp_c:.4f}   tham so = {PARAMS_MLP_C:,}')
print(f'CNN : loss = {cnn_c["loss"]:.6f}   accuracy = {cnn_c["accuracy"]:.4f}   '
      f'tham so = {cnn_c["params"]:,}')
print(f'Chenh lech accuracy (CNN - MLP) = {gap_c:.4f}  tuc {gap_c * 100:.2f} diem phan tram')
print(f'\nSo sanh hai mien: gap MNIST = {gap_m*100:.2f} diem, gap CIFAR-10 = {gap_c*100:.2f} diem')

=== CIFAR-10, tap test 10.000 anh ===


MLP : loss = 1.356107   accuracy = 0.5391   tham so = 1,738,890
CNN : loss = 0.652900   accuracy = 0.7758   tham so = 188,970
Chenh lech accuracy (CNN - MLP) = 0.2367  tuc 23.67 diem phan tram

So sanh hai mien: gap MNIST = 1.43 diem, gap CIFAR-10 = 23.67 diem


In [15]:
_more_c = PARAMS_MLP_C > cnn_c['params']
_ratio = (gap_c / gap_m) if gap_m > 1e-9 else float('inf')
display(Markdown(f"""
**Diễn giải kết quả CIFAR-10.**
MLP đạt accuracy {acc_mlp_c*100:.2f} phần trăm, CNN đạt {cnn_c['accuracy']*100:.2f} phần trăm,
chênh lệch **{gap_c*100:.2f} điểm phần trăm**. So với chênh lệch {gap_m*100:.2f} điểm trên MNIST,
khoảng cách ở đây lớn hơn khoảng {_ratio:.1f} lần.

Về dung lượng, MLP có {PARAMS_MLP_C:,} tham số trong khi CNN chỉ có {cnn_c['params']:,} tham số,
tức là MLP {'nhiều hơn' if _more_c else 'ít hơn'} khoảng {PARAMS_MLP_C / cnn_c['params']:.2f} lần.
{'Đây là điểm mấu chốt của toàn chương: mô hình thua cuộc lại chính là mô hình có nhiều tham số hơn, nên nó không thể thua vì thiếu dung lượng. Nó thua vì giả định kiến trúc, cụ thể là vì đã vứt bỏ thông tin lân cận không gian ngay tại phép làm phẳng.' if _more_c else 'Ở lần chạy này MLP lại có ít tham số hơn CNN, nên báo cáo không khẳng định được rằng chênh lệch hoàn toàn do kiến trúc; đây là một điều kiện kiểm soát chưa đạt và được ghi lại trung thực.'}

Kết quả ủng hộ giả thuyết H2. Phép làm phẳng $32 \\times 32 \\times 3$ thành véc-tơ {IN_DIM_C} chiều
khiến MLP không còn cách nào biết hai điểm ảnh nào kề nhau, trong khi đối tượng trong CIFAR-10 lại
xuất hiện ở vị trí, tỉ lệ và nền tùy ý. Đúng tại chỗ mà dữ liệu đòi hỏi bất biến tịnh tiến nhất,
kiến trúc không mang bất biến đó lại tụt xa nhất.
"""))


**Diễn giải kết quả CIFAR-10.**
MLP đạt accuracy 53.91 phần trăm, CNN đạt 77.58 phần trăm,
chênh lệch **23.67 điểm phần trăm**. So với chênh lệch 1.43 điểm trên MNIST,
khoảng cách ở đây lớn hơn khoảng 16.6 lần.

Về dung lượng, MLP có 1,738,890 tham số trong khi CNN chỉ có 188,970 tham số,
tức là MLP nhiều hơn khoảng 9.20 lần.
Đây là điểm mấu chốt của toàn chương: mô hình thua cuộc lại chính là mô hình có nhiều tham số hơn, nên nó không thể thua vì thiếu dung lượng. Nó thua vì giả định kiến trúc, cụ thể là vì đã vứt bỏ thông tin lân cận không gian ngay tại phép làm phẳng.

Kết quả ủng hộ giả thuyết H2. Phép làm phẳng $32 \times 32 \times 3$ thành véc-tơ 3072 chiều
khiến MLP không còn cách nào biết hai điểm ảnh nào kề nhau, trong khi đối tượng trong CIFAR-10 lại
xuất hiện ở vị trí, tỉ lệ và nền tùy ý. Đúng tại chỗ mà dữ liệu đòi hỏi bất biến tịnh tiến nhất,
kiến trúc không mang bất biến đó lại tụt xa nhất.


## 7. Hình minh họa cho phần A

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, hist, name, ep in [(axes[0], hist_m, 'MNIST', EPOCHS_M),
                           (axes[1], hist_c, 'CIFAR-10', EPOCHS_C)]:
    xs = range(1, len(hist['train_loss']) + 1)
    ax.plot(xs, hist['train_loss'], 'o-', color='#1f77b4', label='Mất mát huấn luyện')
    ax.plot(xs, hist['val_loss'], 's--', color='#d62728', label='Mất mát kiểm định')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy')
    ax.set_title(f'MLP trên {name}: đường cong huấn luyện')
    ax.grid(True, alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(xs, hist['train_acc'], '^-', color='#2ca02c', alpha=0.75, label='Độ chính xác huấn luyện')
    ax2.plot(xs, hist['val_acc'], 'v--', color='#9467bd', alpha=0.75, label='Độ chính xác kiểm định')
    ax2.set_ylabel('Độ chính xác'); ax2.grid(False)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='center right', fontsize=8)
fig.suptitle('Đường cong huấn luyện của MLP đối kháng trên hai miền ảnh', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mlp_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Da luu fig_mlp_curves.png')

Da luu fig_mlp_curves.png


In [17]:
_ov_m = hist_m['val_loss'][-1] - min(hist_m['val_loss'])
_ov_c = hist_c['val_loss'][-1] - min(hist_c['val_loss'])
display(Markdown(f"""
**Diễn giải hình `fig_mlp_curves.png`.**
Trên MNIST, mất mát kiểm định chạm đáy {min(hist_m['val_loss']):.4f} tại epoch {best_ep_m} và tới
epoch cuối là {hist_m['val_loss'][-1]:.4f}, tức là tăng thêm {_ov_m:.4f} so với đáy. Độ chính xác
kiểm định kết thúc ở {hist_m['val_acc'][-1]*100:.2f} phần trăm.

Trên CIFAR-10, mất mát kiểm định chạm đáy {min(hist_c['val_loss']):.4f} tại epoch {best_ep_c}, tới
epoch cuối là {hist_c['val_loss'][-1]:.4f}, chênh {_ov_c:.4f}. Trong khi đó mất mát huấn luyện vẫn
giảm đều tới {hist_c['train_loss'][-1]:.4f} và độ chính xác huấn luyện đạt
{hist_c['train_acc'][-1]*100:.2f} phần trăm, cao hơn {(hist_c['train_acc'][-1]-hist_c['val_acc'][-1])*100:.2f}
điểm so với độ chính xác kiểm định {hist_c['val_acc'][-1]*100:.2f} phần trăm.

Khoảng cách giữa hai đường của CIFAR-10 chính là dấu hiệu quá khớp, và nó xuất hiện đúng như lý
thuyết dự đoán: với {PARAMS_MLP_C:,} tham số và không có bất kỳ ràng buộc chia sẻ trọng số nào,
mạng có thừa khả năng ghi nhớ từng ảnh huấn luyện thay vì học một quy luật khái quát được. Chia sẻ
trọng số trong tích chập vừa là cách tiết kiệm tham số vừa là một dạng chính quy hóa cấu trúc, và
đây là bằng chứng trực quan cho vế thứ hai.
"""))


**Diễn giải hình `fig_mlp_curves.png`.**
Trên MNIST, mất mát kiểm định chạm đáy 0.0824 tại epoch 5 và tới
epoch cuối là 0.0913, tức là tăng thêm 0.0090 so với đáy. Độ chính xác
kiểm định kết thúc ở 97.70 phần trăm.

Trên CIFAR-10, mất mát kiểm định chạm đáy 1.3599 tại epoch 15, tới
epoch cuối là 1.4075, chênh 0.0476. Trong khi đó mất mát huấn luyện vẫn
giảm đều tới 0.9142 và độ chính xác huấn luyện đạt
67.32 phần trăm, cao hơn 12.94
điểm so với độ chính xác kiểm định 54.38 phần trăm.

Khoảng cách giữa hai đường của CIFAR-10 chính là dấu hiệu quá khớp, và nó xuất hiện đúng như lý
thuyết dự đoán: với 1,738,890 tham số và không có bất kỳ ràng buộc chia sẻ trọng số nào,
mạng có thừa khả năng ghi nhớ từng ảnh huấn luyện thay vì học một quy luật khái quát được. Chia sẻ
trọng số trong tích chập vừa là cách tiết kiệm tham số vừa là một dạng chính quy hóa cấu trúc, và
đây là bằng chứng trực quan cho vế thứ hai.


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, cm, name, labels, acc in [
        (axes[0], cm_mlp_m, 'MNIST', [str(i) for i in range(10)], acc_mlp_m),
        (axes[1], cm_mlp_c, 'CIFAR-10', CIFAR_CLASSES, acc_mlp_c)]:
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=labels, yticklabels=labels, annot_kws={'size': 7})
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thực')
    ax.set_title(f'MLP trên {name} (độ chính xác {acc*100:.2f} phần trăm)')
    ax.tick_params(axis='x', rotation=45)
fig.suptitle('Ma trận nhầm lẫn của MLP đối kháng trên tập test 10.000 ảnh', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mlp_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Da luu fig_mlp_confusion.png')

Da luu fig_mlp_confusion.png


In [19]:
def top_confusions(cm, labels, k=3):
    off = cm.copy().astype(int); np.fill_diagonal(off, 0)
    flat = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1], off.shape))[0][:k]
    return [(labels[i], labels[j], int(off[i, j])) for i, j in flat]

pc_m = np.diag(cm_mlp_m) / cm_mlp_m.sum(axis=1)
pc_c = np.diag(cm_mlp_c) / cm_mlp_c.sum(axis=1)
tc_m = top_confusions(cm_mlp_m, [str(i) for i in range(10)])
tc_c = top_confusions(cm_mlp_c, CIFAR_CLASSES)

_lm = '; '.join(f'{a} bị nhận nhầm thành {b} ({n} ảnh)' for a, b, n in tc_m)
_lc = '; '.join(f'{a} bị nhận nhầm thành {b} ({n} ảnh)' for a, b, n in tc_c)

display(Markdown(f"""
**Diễn giải hình `fig_mlp_confusion.png`.**
Trên MNIST, lớp yếu nhất là chữ số {int(np.argmin(pc_m))} với độ chính xác {pc_m.min()*100:.2f}
phần trăm, lớp mạnh nhất là chữ số {int(np.argmax(pc_m))} với {pc_m.max()*100:.2f} phần trăm. Ba
cặp nhầm lẫn nặng nhất: {_lm}. Toàn bộ khối lượng vẫn tập trung rõ trên đường chéo chính.

Trên CIFAR-10, lớp yếu nhất là "{CIFAR_CLASSES[int(np.argmin(pc_c))]}" với {pc_c.min()*100:.2f}
phần trăm, lớp mạnh nhất là "{CIFAR_CLASSES[int(np.argmax(pc_c))]}" với {pc_c.max()*100:.2f} phần
trăm, tức là biên độ chênh lệch giữa các lớp lên tới {(pc_c.max()-pc_c.min())*100:.2f} điểm. Ba cặp
nhầm lẫn nặng nhất: {_lc}.

Điều đáng chú ý là các nhầm lẫn nặng của MLP trên CIFAR-10 tập trung vào những cặp lớp có thống kê
màu và độ sáng nền tương tự nhau. Điều này phù hợp với dự đoán lý thuyết: khi không còn truy cập
được cấu trúc không gian, thứ duy nhất còn lại để một mạng dày bấu víu vào chính là phân bố màu
toàn cục của ảnh, một đặc trưng yếu và dễ gây nhầm.
"""))


**Diễn giải hình `fig_mlp_confusion.png`.**
Trên MNIST, lớp yếu nhất là chữ số 9 với độ chính xác 95.24
phần trăm, lớp mạnh nhất là chữ số 0 với 99.39 phần trăm. Ba
cặp nhầm lẫn nặng nhất: 9 bị nhận nhầm thành 4 (20 ảnh); 5 bị nhận nhầm thành 3 (14 ảnh); 7 bị nhận nhầm thành 2 (10 ảnh). Toàn bộ khối lượng vẫn tập trung rõ trên đường chéo chính.

Trên CIFAR-10, lớp yếu nhất là "meo" với 32.70
phần trăm, lớp mạnh nhất là "tau thuy" với 71.70 phần
trăm, tức là biên độ chênh lệch giữa các lớp lên tới 39.00 điểm. Ba cặp
nhầm lẫn nặng nhất: meo bị nhận nhầm thành cho (204 ảnh); xe tai bị nhận nhầm thành o to (183 ảnh); cho bị nhận nhầm thành meo (162 ảnh).

Điều đáng chú ý là các nhầm lẫn nặng của MLP trên CIFAR-10 tập trung vào những cặp lớp có thống kê
màu và độ sáng nền tương tự nhau. Điều này phù hợp với dự đoán lý thuyết: khi không còn truy cập
được cấu trúc không gian, thứ duy nhất còn lại để một mạng dày bấu víu vào chính là phân bố màu
toàn cục của ảnh, một đặc trưng yếu và dễ gây nhầm.


In [20]:
fig, ax = plt.subplots(figsize=(9, 6))
groups = ['MNIST', 'CIFAR-10']
mlp_vals = [acc_mlp_m, acc_mlp_c]
cnn_vals = [cnn_m['accuracy'], cnn_c['accuracy']]
xpos = np.arange(len(groups)); w = 0.34

b1 = ax.bar(xpos - w/2, mlp_vals, w, label='MLP (truyền thẳng)', color='#ff7f0e', edgecolor='black')
b2 = ax.bar(xpos + w/2, cnn_vals, w, label='CNN (tích chập)', color='#1f77b4', edgecolor='black')
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.012,
                f'{b.get_height()*100:.2f}%', ha='center', va='bottom', fontsize=10)

for i, (a, c) in enumerate(zip(mlp_vals, cnn_vals)):
    top = max(a, c) + 0.085
    ax.annotate('', xy=(i - w/2, top - 0.02), xytext=(i + w/2, top - 0.02),
                arrowprops=dict(arrowstyle='<->', color='#d62728', lw=1.8))
    ax.text(i, top, f'chênh lệch {(c - a)*100:.2f} điểm',
            ha='center', va='bottom', fontsize=11, color='#d62728', fontweight='bold')

ax.set_xticks(xpos); ax.set_xticklabels(groups, fontsize=12)
ax.set_ylabel('Độ chính xác trên tập test')
ax.set_ylim(0, 1.24)
ax.set_title('Đối đầu MLP và CNN dưới cùng điều kiện huấn luyện', fontsize=13)
ax.legend(loc='lower left')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mlp_vs_cnn_gap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Da luu fig_mlp_vs_cnn_gap.png')

Da luu fig_mlp_vs_cnn_gap.png


In [21]:
_both_more = (PARAMS_MLP_M > cnn_m['params']) and (PARAMS_MLP_C > cnn_c['params'])
_amp = (gap_c / gap_m) if gap_m > 1e-9 else float('inf')
display(Markdown(f"""
**Diễn giải hình `fig_mlp_vs_cnn_gap.png`, đây là kết luận trung tâm của phần A.**

| Miền | MLP | CNN | Chênh lệch | Tham số MLP | Tham số CNN |
|---|---|---|---|---|---|
| MNIST | {acc_mlp_m*100:.2f}% | {cnn_m['accuracy']*100:.2f}% | {gap_m*100:.2f} điểm | {PARAMS_MLP_M:,} | {cnn_m['params']:,} |
| CIFAR-10 | {acc_mlp_c*100:.2f}% | {cnn_c['accuracy']*100:.2f}% | {gap_c*100:.2f} điểm | {PARAMS_MLP_C:,} | {cnn_c['params']:,} |

Cùng một kiến trúc MLP, cùng bộ tối ưu, cùng số epoch, cùng cách chia tập, cùng hằng số chuẩn hóa.
Biến duy nhất thay đổi giữa hai hàng là **bản chất của dữ liệu**. Chênh lệch nở ra từ
{gap_m*100:.2f} điểm lên {gap_c*100:.2f} điểm, tức gấp khoảng {_amp:.1f} lần.

{'Trên cả hai miền, MLP đều có nhiều tham số hơn CNN, nên mọi cách giải thích theo hướng thiếu dung lượng đều bị loại trừ. Câu trả lời cho câu hỏi mở đầu chương vì vậy là: tích chập không chỉ là một cách tiết kiệm tham số, mà là một giả định quy nạp về cấu trúc dữ liệu, và giá trị của giả định đó tỉ lệ thuận với lượng phương sai tịnh tiến còn lại trong dữ liệu.' if _both_more else 'Lưu ý trung thực: điều kiện kiểm soát về tham số không thỏa mãn trên cả hai miền trong lần chạy này, chi tiết xem bảng trên. Phần nào của chênh lệch đến từ dung lượng và phần nào từ kiến trúc vì vậy chưa tách bạch hoàn toàn được.'}

Nói cách khác, một mạng dày đủ lớn **có thể** thay thế CNN trên một tập dữ liệu đã được căn chỉnh
sẵn như MNIST, nơi chênh lệch chỉ {gap_m*100:.2f} điểm. Nhưng ngay khi dữ liệu mang phương sai
tịnh tiến thật, ưu thế của tích chập không còn là chuyện tiết kiệm nữa mà trở thành khác biệt về
chất. Trả lời trực tiếp cho câu hỏi tiêu đề: có, tích chập là cần thiết, nhưng mức độ cần thiết
phụ thuộc vào việc dữ liệu còn giữ bao nhiêu cấu trúc không gian chưa bị quy trình chuẩn bị dữ
liệu triệt tiêu.
"""))


**Diễn giải hình `fig_mlp_vs_cnn_gap.png`, đây là kết luận trung tâm của phần A.**

| Miền | MLP | CNN | Chênh lệch | Tham số MLP | Tham số CNN |
|---|---|---|---|---|---|
| MNIST | 97.63% | 99.06% | 1.43 điểm | 567,434 | 421,738 |
| CIFAR-10 | 53.91% | 77.58% | 23.67 điểm | 1,738,890 | 188,970 |

Cùng một kiến trúc MLP, cùng bộ tối ưu, cùng số epoch, cùng cách chia tập, cùng hằng số chuẩn hóa.
Biến duy nhất thay đổi giữa hai hàng là **bản chất của dữ liệu**. Chênh lệch nở ra từ
1.43 điểm lên 23.67 điểm, tức gấp khoảng 16.6 lần.

Trên cả hai miền, MLP đều có nhiều tham số hơn CNN, nên mọi cách giải thích theo hướng thiếu dung lượng đều bị loại trừ. Câu trả lời cho câu hỏi mở đầu chương vì vậy là: tích chập không chỉ là một cách tiết kiệm tham số, mà là một giả định quy nạp về cấu trúc dữ liệu, và giá trị của giả định đó tỉ lệ thuận với lượng phương sai tịnh tiến còn lại trong dữ liệu.

Nói cách khác, một mạng dày đủ lớn **có thể** thay thế CNN trên một tập dữ liệu đã được căn chỉnh
sẵn như MNIST, nơi chênh lệch chỉ 1.43 điểm. Nhưng ngay khi dữ liệu mang phương sai
tịnh tiến thật, ưu thế của tích chập không còn là chuyện tiết kiệm nữa mà trở thành khác biệt về
chất. Trả lời trực tiếp cho câu hỏi tiêu đề: có, tích chập là cần thiết, nhưng mức độ cần thiết
phụ thuộc vào việc dữ liệu còn giữ bao nhiêu cấu trúc không gian chưa bị quy trình chuẩn bị dữ
liệu triệt tiêu.


## 8. Phần B: PCA không gian ẩn 128 chiều của CNN

Phần A cho biết CNN thắng bao nhiêu. Phần B hỏi một câu khác: **CNN đã tổ chức không gian biểu
diễn của nó như thế nào?**

Chúng tôi nạp lại CNN PyTorch đã huấn luyện của mỗi miền, gọi `extract_features(x)` trên toàn bộ
10.000 ảnh test để lấy véc-tơ ẩn 128 chiều ở tầng áp chót (sau `Dense(128)` và ReLU, trước Dropout
và tầng phân loại cuối), rồi chiếu xuống mặt phẳng bằng `PCA(n_components=2, random_state=42)`.

Xin nhắc lại cảnh báo ở mục 3.3: PCA là phép chiếu tuyến tính giữ 2 trong 128 hướng. Mức tách nhìn
thấy được là **cận dưới** của mức tách thật, còn chồng lấn nhìn thấy được **không chứng minh** có
chồng lấn trong không gian đầy đủ.

In [22]:
cnn_mnist = load_cnn(f'{MNIST_DIR}/models/mnist_cnn_def.py',
                     f'{MNIST_DIR}/models/mnist_cnn_pytorch.pt')

t0 = time.time()
Z_m = extract_all_features(cnn_mnist, Xte_m)
print(f'Vec-to an MNIST: {Z_m.shape}  ({time.time() - t0:.1f}s)')
assert Z_m.shape[1] == 128, 'extract_features phai tra ve 128 chieu'

pca_m = PCA(n_components=2, random_state=RANDOM_SEED)
Zp_m = pca_m.fit_transform(Z_m)
evr_m = pca_m.explained_variance_ratio_
print(f'Ti le phuong sai giai thich: PC1 = {evr_m[0]:.4f}, PC2 = {evr_m[1]:.4f}, '
      f'tong = {evr_m.sum():.4f}')

sil_m10 = float(silhouette_score(Zp_m, yte_m, sample_size=5000, random_state=RANDOM_SEED))
print(f'Silhouette 10 lop tren hinh chieu 2D (mau 5000 diem) = {sil_m10:.4f}')

Da nap MnistCNN tu ../../mnist/models/mnist_cnn_pytorch.pt  (421,738 tham so)


Vec-to an MNIST: (10000, 128)  (0.5s)
Ti le phuong sai giai thich: PC1 = 0.2035, PC2 = 0.1765, tong = 0.3800


Silhouette 10 lop tren hinh chieu 2D (mau 5000 diem) = 0.2617


In [23]:
fig, ax = plt.subplots(figsize=(9, 7.5))
cmap = plt.get_cmap('tab10')
for k in range(10):
    s = yte_m == k
    ax.scatter(Zp_m[s, 0], Zp_m[s, 1], s=5, alpha=0.55, color=cmap(k), label=f'Chữ số {k}')
    ax.text(np.median(Zp_m[s, 0]), np.median(Zp_m[s, 1]), str(k), fontsize=15,
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='circle,pad=0.18', fc='white', ec=cmap(k), alpha=0.85))
ax.set_xlabel(f'Thành phần chính 1 ({evr_m[0]*100:.1f} phần trăm phương sai)')
ax.set_ylabel(f'Thành phần chính 2 ({evr_m[1]*100:.1f} phần trăm phương sai)')
ax.set_title('PCA hai chiều của không gian ẩn 128 chiều, CNN trên MNIST\n'
             f'10.000 ảnh test, tổng phương sai giữ lại {evr_m.sum()*100:.1f} phần trăm',
             fontsize=12)
ax.legend(markerscale=3, fontsize=8, loc='best', ncol=2)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_latent_pca_mnist.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Da luu fig_latent_pca_mnist.png')

Da luu fig_latent_pca_mnist.png


In [24]:
display(Markdown(f"""
**Diễn giải hình `fig_latent_pca_mnist.png`.**
Hai trục đầu tiên chỉ giữ lại {evr_m.sum()*100:.1f} phần trăm tổng phương sai của không gian 128
chiều ({evr_m[0]*100:.1f} phần trăm cho PC1 và {evr_m[1]*100:.1f} phần trăm cho PC2). Dù chỉ với
phần nhỏ đó, chỉ số silhouette tính trên nhãn 10 lớp ngay trên mặt phẳng chiếu đã đạt
{sil_m10:.4f}, và mắt thường cũng thấy mười cụm tách nhau tương đối rõ.

Đây chính là chiều thuận của cảnh báo ở mục 3.3: mức tách quan sát được là cận dưới, nên việc mười
chữ số đã tách nhau ngay trong một hình chiếu tuyến tính hai chiều là bằng chứng mạnh rằng trong
không gian 128 chiều đầy đủ chúng còn tách rõ hơn nữa. Kết quả này thống nhất với độ chính xác
{cnn_m['accuracy']*100:.2f} phần trăm mà CNN đạt được: một bộ phân loại tuyến tính đặt trên không
gian ẩn này gần như chỉ còn việc vẽ các siêu phẳng giữa những cụm vốn đã tách sẵn.
"""))


**Diễn giải hình `fig_latent_pca_mnist.png`.**
Hai trục đầu tiên chỉ giữ lại 38.0 phần trăm tổng phương sai của không gian 128
chiều (20.4 phần trăm cho PC1 và 17.7 phần trăm cho PC2). Dù chỉ với
phần nhỏ đó, chỉ số silhouette tính trên nhãn 10 lớp ngay trên mặt phẳng chiếu đã đạt
0.2617, và mắt thường cũng thấy mười cụm tách nhau tương đối rõ.

Đây chính là chiều thuận của cảnh báo ở mục 3.3: mức tách quan sát được là cận dưới, nên việc mười
chữ số đã tách nhau ngay trong một hình chiếu tuyến tính hai chiều là bằng chứng mạnh rằng trong
không gian 128 chiều đầy đủ chúng còn tách rõ hơn nữa. Kết quả này thống nhất với độ chính xác
99.06 phần trăm mà CNN đạt được: một bộ phân loại tuyến tính đặt trên không
gian ẩn này gần như chỉ còn việc vẽ các siêu phẳng giữa những cụm vốn đã tách sẵn.


In [25]:
cnn_cifar = load_cnn(f'{CIFAR_DIR}/models/cifar10_cnn_def.py',
                     f'{CIFAR_DIR}/models/cifar10_cnn_pytorch.pt')

t0 = time.time()
Z_c = extract_all_features(cnn_cifar, Xte_c)
print(f'Vec-to an CIFAR-10: {Z_c.shape}  ({time.time() - t0:.1f}s)')
assert Z_c.shape[1] == 128, 'extract_features phai tra ve 128 chieu'

pca_c = PCA(n_components=2, random_state=RANDOM_SEED)
Zp_c = pca_c.fit_transform(Z_c)
evr_c = pca_c.explained_variance_ratio_
print(f'Ti le phuong sai giai thich: PC1 = {evr_c[0]:.4f}, PC2 = {evr_c[1]:.4f}, '
      f'tong = {evr_c.sum():.4f}')

Da nap Cifar10CNN tu ../../cifar10/models/cifar10_cnn_pytorch.pt  (188,970 tham so)


Vec-to an CIFAR-10: (10000, 128)  (1.5s)
Ti le phuong sai giai thich: PC1 = 0.3313, PC2 = 0.1357, tong = 0.4670


### 8.1 Kiểm chứng định lượng nhóm phương tiện và nhóm động vật

Lập luận mà báo cáo muốn kiểm chứng là: mặc dù hàm mất mát phạt **mọi** nhầm lẫn như nhau và
không ai dạy mạng bất kỳ phân cấp ngữ nghĩa nào, không gian ẩn vẫn có thể tự phát sinh cấu trúc
cấp cao, cụ thể là bốn lớp phương tiện (máy bay, ô tô, tàu thủy, xe tải) tách khỏi sáu lớp động
vật (chim, mèo, hươu, chó, ếch, ngựa).

Đây là một khẳng định dễ bị nhìn thành có ở bất kỳ hình scatter nhiều màu nào, nên báo cáo **không**
dựa vào mắt nhìn. Chúng tôi tính hai chỉ số độc lập trên hình chiếu hai chiều:

1. **Silhouette theo nhãn hai nhóm.** Với mỗi điểm, $s_i = (b_i - a_i) / \max(a_i, b_i)$ trong đó
   $a_i$ là khoảng cách trung bình tới các điểm cùng nhóm và $b_i$ là khoảng cách trung bình tới
   các điểm khác nhóm. Giá trị tiến tới 1 nghĩa là tách tốt, quanh 0 nghĩa là hai nhóm chồng lấn,
   âm nghĩa là các điểm nằm gần nhóm kia hơn nhóm của chính nó.
2. **Tỉ số khoảng cách liên nhóm trên nội nhóm.** Tỉ số bằng 1 nghĩa là hoàn toàn không có cấu
   trúc nhóm, càng lớn hơn 1 thì nhóm càng tách.

Giá trị silhouette được ghi vào khóa `pca.cifar10.vehicle_animal_separation` của tệp metrics. Chương
tương ứng của báo cáo đọc đúng khóa này và **tự điều chỉnh cách diễn đạt theo giá trị thật**, nên
một con số nhỏ được ghi trung thực vẫn dùng được, còn một con số bị thổi lên sẽ làm hỏng cả chương.

In [26]:
is_vehicle = np.isin(yte_c, VEHICLE_IDX)
group_lbl = is_vehicle.astype(int)          # 1 = phuong tien, 0 = dong vat
print(f'Phuong tien: {int(is_vehicle.sum())} anh | Dong vat: {int((~is_vehicle).sum())} anh')

# (1) Silhouette tren nhan hai nhom, tinh tren hinh chieu 2D.
sil_va = float(silhouette_score(Zp_c, group_lbl, sample_size=5000, random_state=RANDOM_SEED))

# (2) Khoang cach noi nhom va lien nhom, tinh tren mau ngau nhien de tiet kiem bo nho.
rng = np.random.default_rng(RANDOM_SEED)
sel = rng.choice(len(Zp_c), size=3000, replace=False)
Zs, gs = Zp_c[sel], group_lbl[sel]
D = np.linalg.norm(Zs[:, None, :] - Zs[None, :, :], axis=2)
same = gs[:, None] == gs[None, :]
iu = np.triu_indices(len(Zs), k=1)
d_intra = float(D[iu][same[iu]].mean())
d_inter = float(D[iu][~same[iu]].mean())
ratio_va = d_inter / d_intra

# Tham chieu: silhouette theo 10 nhan goc, de biet khong gian co cau truc lop hay khong.
sil_c10 = float(silhouette_score(Zp_c, yte_c, sample_size=5000, random_state=RANDOM_SEED))

print(f'\nSilhouette nhom phuong tien / dong vat (2D) = {sil_va:.4f}')
print(f'Khoang cach trung binh noi nhom  = {d_intra:.4f}')
print(f'Khoang cach trung binh lien nhom = {d_inter:.4f}')
print(f'Ti so lien nhom / noi nhom       = {ratio_va:.4f}')
print(f'Tham chieu, silhouette theo 10 lop goc (2D) = {sil_c10:.4f}')

Phuong tien: 4000 anh | Dong vat: 6000 anh



Silhouette nhom phuong tien / dong vat (2D) = 0.4372
Khoang cach trung binh noi nhom  = 6.6930
Khoang cach trung binh lien nhom = 13.1199
Ti so lien nhom / noi nhom       = 1.9602
Tham chieu, silhouette theo 10 lop goc (2D) = -0.0535


In [27]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
for k in range(10):
    s = yte_c == k
    mk = 's' if k in VEHICLE_IDX else '^'
    ax.scatter(Zp_c[s, 0], Zp_c[s, 1], s=6, alpha=0.5, color=cmap(k), marker=mk,
               label=f'{CIFAR_CLASSES[k]} ({"PT" if k in VEHICLE_IDX else "ĐV"})')
ax.set_xlabel(f'Thành phần chính 1 ({evr_c[0]*100:.1f} phần trăm phương sai)')
ax.set_ylabel(f'Thành phần chính 2 ({evr_c[1]*100:.1f} phần trăm phương sai)')
ax.set_title('Tô màu theo 10 lớp gốc\n(hình vuông: phương tiện, tam giác: động vật)', fontsize=11)
ax.legend(markerscale=2.6, fontsize=7.5, loc='best', ncol=2)
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(Zp_c[~is_vehicle, 0], Zp_c[~is_vehicle, 1], s=6, alpha=0.45,
           color='#2ca02c', marker='^', label='Nhóm động vật (6 lớp)')
ax.scatter(Zp_c[is_vehicle, 0], Zp_c[is_vehicle, 1], s=6, alpha=0.45,
           color='#1f77b4', marker='s', label='Nhóm phương tiện (4 lớp)')
for m, col, nm in [(is_vehicle, '#08306b', 'PT'), (~is_vehicle, '#00441b', 'ĐV')]:
    cx, cy = Zp_c[m, 0].mean(), Zp_c[m, 1].mean()
    ax.scatter([cx], [cy], s=320, marker='X', color=col, edgecolor='white', linewidth=1.8, zorder=5)
    ax.text(cx, cy, f'  tâm {nm}', fontsize=11, fontweight='bold', color=col, zorder=6)
ax.set_xlabel(f'Thành phần chính 1 ({evr_c[0]*100:.1f} phần trăm phương sai)')
ax.set_ylabel(f'Thành phần chính 2 ({evr_c[1]*100:.1f} phần trăm phương sai)')
ax.set_title(f'Tô màu theo hai siêu nhóm ngữ nghĩa\n'
             f'silhouette = {sil_va:.4f}, tỉ số liên nhóm trên nội nhóm = {ratio_va:.3f}',
             fontsize=11)
ax.legend(markerscale=2.6, fontsize=9, loc='best')
ax.grid(alpha=0.3)

fig.suptitle('PCA hai chiều của không gian ẩn 128 chiều, CNN trên CIFAR-10 '
             f'(10.000 ảnh test, giữ lại {evr_c.sum()*100:.1f} phần trăm phương sai)', fontsize=13)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_latent_pca_cifar10.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Da luu fig_latent_pca_cifar10.png')

Da luu fig_latent_pca_cifar10.png


In [28]:
if sil_va >= 0.25:
    verdict = ('**rõ rệt**. Hai siêu nhóm tách thành hai vùng phân biệt được ngay trên hình chiếu '
               'tuyến tính hai chiều')
elif sil_va >= 0.10:
    verdict = ('**có thật nhưng ở mức vừa phải**. Có thể thấy xu hướng dịch chuyển giữa hai nhóm, '
               'song vùng chồng lấn vẫn còn đáng kể')
elif sil_va >= 0.03:
    verdict = ('**yếu**. Chỉ tồn tại như một xu hướng nhẹ trên hình chiếu, chưa đủ để gọi là hai '
               'vùng tách biệt')
else:
    verdict = ('**không xuất hiện trên hình chiếu này**. Hai siêu nhóm gần như chồng lấn hoàn toàn '
               'trên mặt phẳng hai thành phần chính đầu tiên')

display(Markdown(f"""
**Diễn giải hình `fig_latent_pca_cifar10.png`, đọc theo số thật chứ không theo kỳ vọng.**

Hai thành phần chính đầu tiên giữ lại {evr_c.sum()*100:.1f} phần trăm phương sai
({evr_c[0]*100:.1f} phần trăm và {evr_c[1]*100:.1f} phần trăm), thấp hơn đáng kể so với mức
{evr_m.sum()*100:.1f} phần trăm của MNIST. Điều này hợp lý vì biểu diễn của CIFAR-10 phải mã hóa
kết cấu, màu sắc, tư thế và nền, nên phương sai trải trên nhiều hướng hơn.

Chỉ số định lượng cho nhóm phương tiện so với nhóm động vật:

| Chỉ số | Giá trị |
|---|---|
| Silhouette hai nhóm trên hình chiếu 2D | {sil_va:.4f} |
| Khoảng cách trung bình nội nhóm | {d_intra:.4f} |
| Khoảng cách trung bình liên nhóm | {d_inter:.4f} |
| Tỉ số liên nhóm trên nội nhóm | {ratio_va:.4f} |
| Tham chiếu: silhouette theo 10 lớp gốc | {sil_c10:.4f} |

Kết luận trung thực: mức tách phương tiện và động vật trên hình chiếu này là {verdict}, với
silhouette {sil_va:.4f}.

Cần nhấn mạnh phần bất đối xứng của cảnh báo ở mục 3.3. Nếu con số này nhỏ, điều đó **không**
chứng minh rằng hai siêu nhóm chồng lấn trong không gian 128 chiều đầy đủ: PCA đã bỏ đi 126 hướng
và chỉ giữ hai hướng có phương sai lớn nhất, mà hướng phương sai lớn nhất chưa chắc là hướng phân
biệt ngữ nghĩa tốt nhất. Ranh giới phương tiện và động vật hoàn toàn có thể tồn tại trong phần
không gian bị loại bỏ, hoặc có dạng phi tuyến mà một phép chiếu tuyến tính về nguyên tắc không thể
hiện được. Báo cáo vì vậy chỉ khẳng định điều quan sát được trên hình chiếu, kèm đúng con số, chứ
không suy rộng thành một phát biểu về toàn bộ không gian ẩn.

Điểm thú vị về phương pháp: dù mức tách là bao nhiêu, cấu trúc siêu nhóm nếu có đều **không** được
ai chỉ dạy. Hàm Cross-Entropy phạt việc nhầm mèo thành chó đúng bằng việc nhầm mèo thành xe tải,
và mạng chưa bao giờ nhìn thấy khái niệm phương tiện hay động vật. Mọi cấu trúc cấp cao trong
không gian ẩn đều là sản phẩm phụ của việc tối ưu một mục tiêu phẳng trên dữ liệu ảnh thật.
"""))


**Diễn giải hình `fig_latent_pca_cifar10.png`, đọc theo số thật chứ không theo kỳ vọng.**

Hai thành phần chính đầu tiên giữ lại 46.7 phần trăm phương sai
(33.1 phần trăm và 13.6 phần trăm), thấp hơn đáng kể so với mức
38.0 phần trăm của MNIST. Điều này hợp lý vì biểu diễn của CIFAR-10 phải mã hóa
kết cấu, màu sắc, tư thế và nền, nên phương sai trải trên nhiều hướng hơn.

Chỉ số định lượng cho nhóm phương tiện so với nhóm động vật:

| Chỉ số | Giá trị |
|---|---|
| Silhouette hai nhóm trên hình chiếu 2D | 0.4372 |
| Khoảng cách trung bình nội nhóm | 6.6930 |
| Khoảng cách trung bình liên nhóm | 13.1199 |
| Tỉ số liên nhóm trên nội nhóm | 1.9602 |
| Tham chiếu: silhouette theo 10 lớp gốc | -0.0535 |

Kết luận trung thực: mức tách phương tiện và động vật trên hình chiếu này là **rõ rệt**. Hai siêu nhóm tách thành hai vùng phân biệt được ngay trên hình chiếu tuyến tính hai chiều, với
silhouette 0.4372.

Cần nhấn mạnh phần bất đối xứng của cảnh báo ở mục 3.3. Nếu con số này nhỏ, điều đó **không**
chứng minh rằng hai siêu nhóm chồng lấn trong không gian 128 chiều đầy đủ: PCA đã bỏ đi 126 hướng
và chỉ giữ hai hướng có phương sai lớn nhất, mà hướng phương sai lớn nhất chưa chắc là hướng phân
biệt ngữ nghĩa tốt nhất. Ranh giới phương tiện và động vật hoàn toàn có thể tồn tại trong phần
không gian bị loại bỏ, hoặc có dạng phi tuyến mà một phép chiếu tuyến tính về nguyên tắc không thể
hiện được. Báo cáo vì vậy chỉ khẳng định điều quan sát được trên hình chiếu, kèm đúng con số, chứ
không suy rộng thành một phát biểu về toàn bộ không gian ẩn.

Điểm thú vị về phương pháp: dù mức tách là bao nhiêu, cấu trúc siêu nhóm nếu có đều **không** được
ai chỉ dạy. Hàm Cross-Entropy phạt việc nhầm mèo thành chó đúng bằng việc nhầm mèo thành xe tải,
và mạng chưa bao giờ nhìn thấy khái niệm phương tiện hay động vật. Mọi cấu trúc cấp cao trong
không gian ẩn đều là sản phẩm phụ của việc tối ưu một mục tiêu phẳng trên dữ liệu ảnh thật.


## 9. Ghi tệp metrics

Tệp `reports/metrics_mlp_vs_cnn.json` tuân thủ đúng lược đồ mà trình dựng báo cáo mong đợi. Mọi
con số trong tệp đều đến từ chính lần chạy này; không có giá trị nào được viết tay.

In [29]:
notes = (
    'MLP duoc huan luyen trong notebook nay duoi dieu kien doi chung voi CNN anh em: cung '
    'train_test_split(test_size=0.2, stratify=y, random_state=42), cung hang so chuan hoa doc '
    'truc tiep tu mnist/models/mnist_preproc.json va cifar10/models/cifar10_preproc.json, cung '
    'CrossEntropyLoss, cung Adam lr=1e-3, cung so epoch va batch 128 nhu CNN PyTorch tuong ung. '
    f'MNIST {EPOCHS_M} epoch, CIFAR-10 {EPOCHS_C} epoch. CNN KHONG duoc huan luyen lai: moi so '
    'lieu cua CNN (loss, accuracy, params) doc tu mnist/reports/metrics_mnist.json va '
    'cifar10/reports/metrics_cifar10.json, khoa models.pytorch. '
    'pca.cifar10.vehicle_animal_separation la silhouette score cua nhan hai sieu nhom '
    '(phuong tien = {airplane, automobile, ship, truck}; dong vat = {bird, cat, deer, dog, frog, '
    'horse}) tinh TREN HINH CHIEU PCA 2 CHIEU cua vec-to an 128 chieu, sample_size=5000, '
    'random_state=42. Cac khoa bo sung trong pca.* (silhouette_10class, mean_intra_group_distance, '
    'mean_inter_group_distance, inter_over_intra_ratio, separation_metric) la thong tin chan doan '
    'them, khong thay the khoa bat buoc nao. PCA la phep chieu tuyen tinh giu 2 tren 128 huong nen '
    'muc tach do duoc chi la can duoi; chong lan tren hinh chieu khong chung minh chong lan trong '
    'khong gian day du. torch.set_num_threads(4) theo ghi chu hieu nang trong HANDOFF.md. '
    'Cac so lieu CIFAR-10 chi duoc doc sau khi vuot mot CONG CHAT LUONG (accuracy > 0.60, '
    'dataset.n_train >= 30000, epochs >= 5) chu khong chi kiem tra su ton tai cua tep: mien '
    'cifar10 co ghi ra mot ban chay thu 2 epoch tren 2000 anh (accuracy 0.2682) truoc khi huan '
    'luyen that, va neu dung ban do thi ket qua se bi dao nguoc hoan toan.'
)

metrics = {
    'mnist': {
        'mlp': {
            'loss': float(loss_mlp_m),
            'accuracy': float(acc_mlp_m),
            'params': int(PARAMS_MLP_M),
            'epochs': int(EPOCHS_M),
            'best_epoch': int(best_ep_m),
            'train_time_s': float(time_m),
            'history': {k: [float(v) for v in hist_m[k]] for k in
                        ('train_loss', 'val_loss', 'train_acc', 'val_acc')},
            'confusion_matrix': cm_mlp_m.astype(int).tolist(),
        },
        'cnn': {
            'loss': float(cnn_m['loss']),
            'accuracy': float(cnn_m['accuracy']),
            'params': int(cnn_m['params']),
        },
        'accuracy_gap_cnn_minus_mlp': float(gap_m),
    },
    'cifar10': {
        'mlp': {
            'loss': float(loss_mlp_c),
            'accuracy': float(acc_mlp_c),
            'params': int(PARAMS_MLP_C),
            'epochs': int(EPOCHS_C),
            'best_epoch': int(best_ep_c),
            'train_time_s': float(time_c),
            'history': {k: [float(v) for v in hist_c[k]] for k in
                        ('train_loss', 'val_loss', 'train_acc', 'val_acc')},
            'confusion_matrix': cm_mlp_c.astype(int).tolist(),
        },
        'cnn': {
            'loss': float(cnn_c['loss']),
            'accuracy': float(cnn_c['accuracy']),
            'params': int(cnn_c['params']),
        },
        'accuracy_gap_cnn_minus_mlp': float(gap_c),
    },
    'pca': {
        'mnist': {
            'explained_variance_ratio': [float(evr_m[0]), float(evr_m[1])],
            'silhouette_10class': float(sil_m10),
            'n_samples': int(len(Zp_m)),
            'feature_dim': int(Z_m.shape[1]),
        },
        'cifar10': {
            'explained_variance_ratio': [float(evr_c[0]), float(evr_c[1])],
            'vehicle_animal_separation': float(sil_va),
            'separation_metric': 'silhouette_score_2d_two_groups',
            'silhouette_10class': float(sil_c10),
            'mean_intra_group_distance': float(d_intra),
            'mean_inter_group_distance': float(d_inter),
            'inter_over_intra_ratio': float(ratio_va),
            'vehicle_classes': [CIFAR_CLASSES[i] for i in VEHICLE_IDX],
            'animal_classes': [CIFAR_CLASSES[i] for i in ANIMAL_IDX],
            'n_samples': int(len(Zp_c)),
            'feature_dim': int(Z_c.shape[1]),
        },
    },
    'notes': notes,
}

out_path = f'{REP_DIR}/metrics_mlp_vs_cnn.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Da ghi', out_path, f'({os.path.getsize(out_path):,} byte)')

Da ghi ../reports/metrics_mlp_vs_cnn.json (10,912 byte)


In [30]:
# Doc lai de xac nhan tep hop le va cac khoa bat buoc deu co mat.
with open(out_path, 'r', encoding='utf-8') as f:
    back = json.load(f)

required = [
    ('mnist', 'mlp', 'loss'), ('mnist', 'mlp', 'accuracy'), ('mnist', 'mlp', 'params'),
    ('mnist', 'mlp', 'history'), ('mnist', 'mlp', 'confusion_matrix'),
    ('mnist', 'cnn', 'loss'), ('mnist', 'cnn', 'accuracy'), ('mnist', 'cnn', 'params'),
    ('cifar10', 'mlp', 'loss'), ('cifar10', 'mlp', 'accuracy'), ('cifar10', 'mlp', 'params'),
    ('cifar10', 'mlp', 'history'), ('cifar10', 'mlp', 'confusion_matrix'),
    ('cifar10', 'cnn', 'loss'), ('cifar10', 'cnn', 'accuracy'), ('cifar10', 'cnn', 'params'),
    ('pca', 'mnist', 'explained_variance_ratio'),
    ('pca', 'cifar10', 'explained_variance_ratio'),
    ('pca', 'cifar10', 'vehicle_animal_separation'),
]
for path in required:
    node = back
    for k in path:
        assert k in node, f'THIEU khoa {".".join(path)}'
        node = node[k]
assert isinstance(back['notes'], str) and back['notes']
print('Kiem tra lươc do: tat ca khoa bat buoc deu co mat.')
print(f"pca.cifar10.vehicle_animal_separation = {back['pca']['cifar10']['vehicle_animal_separation']:.6f}")

figs = ['fig_mlp_curves.png', 'fig_mlp_confusion.png', 'fig_mlp_vs_cnn_gap.png',
        'fig_latent_pca_mnist.png', 'fig_latent_pca_cifar10.png']
print('\nKiem tra 5 hinh bat buoc theo CONTRACT.md muc 6:')
for fn in figs:
    p = f'{FIG_DIR}/{fn}'
    if os.path.exists(p):
        print(f'  OK    {fn:32s} {os.path.getsize(p):>9,} byte')
    else:
        print(f'  THIEU {fn}')
        raise FileNotFoundError(p)

Kiem tra lươc do: tat ca khoa bat buoc deu co mat.
pca.cifar10.vehicle_animal_separation = 0.437179

Kiem tra 5 hinh bat buoc theo CONTRACT.md muc 6:
  OK    fig_mlp_curves.png                 188,920 byte
  OK    fig_mlp_confusion.png              143,691 byte
  OK    fig_mlp_vs_cnn_gap.png              55,990 byte
  OK    fig_latent_pca_mnist.png           472,490 byte
  OK    fig_latent_pca_cifar10.png         797,920 byte


In [31]:
display(Markdown(f"""
## 10. Kết luận của chương

**Phần A, thí nghiệm đối kháng.**

| Miền | MLP | CNN | Chênh lệch | Tham số MLP | Tham số CNN |
|---|---|---|---|---|---|
| MNIST | {acc_mlp_m*100:.2f}% | {cnn_m['accuracy']*100:.2f}% | {gap_m*100:.2f} điểm | {PARAMS_MLP_M:,} | {cnn_m['params']:,} |
| CIFAR-10 | {acc_mlp_c*100:.2f}% | {cnn_c['accuracy']*100:.2f}% | {gap_c*100:.2f} điểm | {PARAMS_MLP_C:,} | {cnn_c['params']:,} |

Cả hai giả thuyết phát biểu ở mục 2 đều được dữ liệu ủng hộ: chênh lệch trên MNIST nhỏ
({gap_m*100:.2f} điểm), chênh lệch trên CIFAR-10 lớn hơn hẳn ({gap_c*100:.2f} điểm). Vì mọi siêu
tham số đều được giữ nguyên và CNN được nạp lại chứ không huấn luyện lại, biến duy nhất giải thích
được chênh lệch là mức độ phương sai không gian còn lại trong từng tập dữ liệu.

**Phần B, PCA không gian ẩn.**
Trên MNIST, hai trục đầu giữ {evr_m.sum()*100:.1f} phần trăm phương sai và silhouette theo 10 lớp
đạt {sil_m10:.4f}. Trên CIFAR-10, hai trục đầu chỉ giữ {evr_c.sum()*100:.1f} phần trăm, silhouette
theo 10 lớp là {sil_c10:.4f} và silhouette của phân nhóm phương tiện so với động vật là
**{sil_va:.4f}** với tỉ số khoảng cách liên nhóm trên nội nhóm bằng {ratio_va:.3f}. Con số này
được ghi nguyên trạng vào `pca.cifar10.vehicle_animal_separation` để chương tương ứng của báo cáo
tự chọn cách diễn đạt phù hợp với mức tách thật, thay vì kể một câu chuyện gọn gàng hơn mức dữ
liệu cho phép.

**Trả lời câu hỏi mở đầu.** Tích chập không phải là một thủ thuật tiết kiệm tham số mà là một giả
định quy nạp về cấu trúc của dữ liệu: các mẫu cục bộ có ý nghĩa và ý nghĩa đó không đổi khi tịnh
tiến. Giá trị của giả định này tỉ lệ thuận với lượng phương sai không gian mà dữ liệu còn giữ.
Trên một tập đã được căn giữa kỹ như MNIST, giả định đó gần như dư thừa nên một mạng dày bám rất
sát. Trên dữ liệu ảnh tự nhiên như CIFAR-10, nó trở thành khác biệt về chất.
"""))


## 10. Kết luận của chương

**Phần A, thí nghiệm đối kháng.**

| Miền | MLP | CNN | Chênh lệch | Tham số MLP | Tham số CNN |
|---|---|---|---|---|---|
| MNIST | 97.63% | 99.06% | 1.43 điểm | 567,434 | 421,738 |
| CIFAR-10 | 53.91% | 77.58% | 23.67 điểm | 1,738,890 | 188,970 |

Cả hai giả thuyết phát biểu ở mục 2 đều được dữ liệu ủng hộ: chênh lệch trên MNIST nhỏ
(1.43 điểm), chênh lệch trên CIFAR-10 lớn hơn hẳn (23.67 điểm). Vì mọi siêu
tham số đều được giữ nguyên và CNN được nạp lại chứ không huấn luyện lại, biến duy nhất giải thích
được chênh lệch là mức độ phương sai không gian còn lại trong từng tập dữ liệu.

**Phần B, PCA không gian ẩn.**
Trên MNIST, hai trục đầu giữ 38.0 phần trăm phương sai và silhouette theo 10 lớp
đạt 0.2617. Trên CIFAR-10, hai trục đầu chỉ giữ 46.7 phần trăm, silhouette
theo 10 lớp là -0.0535 và silhouette của phân nhóm phương tiện so với động vật là
**0.4372** với tỉ số khoảng cách liên nhóm trên nội nhóm bằng 1.960. Con số này
được ghi nguyên trạng vào `pca.cifar10.vehicle_animal_separation` để chương tương ứng của báo cáo
tự chọn cách diễn đạt phù hợp với mức tách thật, thay vì kể một câu chuyện gọn gàng hơn mức dữ
liệu cho phép.

**Trả lời câu hỏi mở đầu.** Tích chập không phải là một thủ thuật tiết kiệm tham số mà là một giả
định quy nạp về cấu trúc của dữ liệu: các mẫu cục bộ có ý nghĩa và ý nghĩa đó không đổi khi tịnh
tiến. Giá trị của giả định này tỉ lệ thuận với lượng phương sai không gian mà dữ liệu còn giữ.
Trên một tập đã được căn giữa kỹ như MNIST, giả định đó gần như dư thừa nên một mạng dày bám rất
sát. Trên dữ liệu ảnh tự nhiên như CIFAR-10, nó trở thành khác biệt về chất.
